# Label Raw ADS-B Data

This notebook generates pushback labels for one airport and season.

The pushback labeling logic is shared across all airport/season combinations. Dataset-specific quality-control decisions are stored in separate YAML configuration files containing automatic filter thresholds and manually reviewed flight IDs.

## 1. Imports

In [1]:
from pathlib import Path
import warnings

import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yaml

from IPython.display import clear_output, display
from traffic.core import Traffic

warnings.filterwarnings("ignore", category=RuntimeWarning)

## 2. Configuration

Select the airport and season. The same handle controls the raw-data input path, output paths, and the corresponding review configuration.

Expected raw-data location:

`data/train_validation/01_source/raw/<AIRPORT>/<SEASON>/`

Airport/season review configuration:

`01_data_generation/01_train_validation/02_label_raw_data/configs/<AIRPORT>/<SEASON>.yaml`

In [2]:
airport_str = "LSZH"
season = "Spring"  # Spring, Summer, Autumn, or Winter

rebuild_preprocessed = False
max_workers = 10

valid_seasons = {"Spring", "Summer", "Autumn", "Winter"}

if season not in valid_seasons:
    raise ValueError(
        f"Invalid season '{season}'. Expected one of: {sorted(valid_seasons)}"
    )


# Project paths
PROJECT_ROOT = Path.home() / "Projects" / "OSN_pushback_detection"
DATA_ROOT = PROJECT_ROOT / "data"

LABELING_ROOT = (
    PROJECT_ROOT
    / "01_data_generation"
    / "01_train_validation"
    / "02_label_raw_data"
)


# Raw data
raw_dir = (
    DATA_ROOT
    / "train_validation"
    / "01_source"
    / "raw"
    / airport_str
    / season
)


# Preprocessed seasonal dataset
preprocessed_dir = (
    DATA_ROOT
    / "train_validation"
    / "02_preprocessed"
    / airport_str
    / season
)

preprocessed_dir.mkdir(parents=True, exist_ok=True)

preprocessed_path = (
    preprocessed_dir
    / f"{airport_str}_{season}_preprocessed.parquet"
)


# Labels after automatic filtering and manual review
reviewed_dir = (
    DATA_ROOT
    / "train_validation"
    / "03_reviewed"
    / airport_str
    / season
)

reviewed_dir.mkdir(parents=True, exist_ok=True)


# Airport- and season-specific review configuration
config_path = (
    LABELING_ROOT
    / "configs"
    / airport_str
    / f"{season}.yaml"
)

if config_path.exists():
    with config_path.open("r", encoding="utf-8") as file:
        review_config = yaml.safe_load(file) or {}

    print(f"Loaded review configuration: {config_path}")

else:
    review_config = {
        "airport": airport_str,
        "season": season,
        "filters": {
            "max_distance": None,
            "max_distance_start": None,
            "min_duration": None,
            "min_average_speed": None,
        },
        "manual_review": {
            "pushbacks_remove": [],
            "pushbacks_keep": [],
            "no_pushbacks_remove": [],
        },
    }

    print(
        f"No review configuration found for {airport_str} {season}. "
        "Starting with an empty review configuration."
    )

review_config

Loaded review configuration: /cluster/home/hanv/Projects/OSN_pushback_detection/01_data_generation/01_train_validation/02_label_raw_data/configs/LSZH/Spring.yaml


{'airport': 'LSZH',
 'season': 'Spring',
 'filters': {'max_distance': 0.5,
  'max_distance_start': 0.1,
  'min_duration': 17,
  'min_average_speed': 0.00017},
 'reasons': {'max_distance': 'Pushback segments with a distance greater than 0.5 NM were found to be incorrectly labeled.\n',
  'max_distance_start': 'Pushback segments starting more than 0.1 NM from the beginning of the trajectory were found to be incorrectly labeled, affected by noise, or incomplete.\n',
  'min_duration': 'Pushback segments shorter than 17 seconds were found to be incorrectly labeled or to have an incorrectly detected start or end.\n',
  'min_average_speed': 'Pushback segments with an average speed below 0.000170 were mostly found to be incorrectly labeled.\n',
  'pushbacks_remove': 'Additional incorrectly labeled pushback segments identified during manual inspection.\n',
  'pushbacks_keep': 'Automatically filtered pushback segments that were manually confirmed to be correctly labeled.\n',
  'no_pushbacks_remov

## 3. Load or Build the Seasonal Dataset

The labeling workflow uses one combined Parquet file per airport and season.

If the preprocessed seasonal Parquet already exists and `rebuild_preprocessed` is `False`, it is loaded directly. This preserves the previously assigned flight IDs across repeated labeling and review sessions.

If the file does not exist, or if `rebuild_preprocessed` is set to `True`, the hourly raw-data files are combined, the required data types are normalized, flight IDs are assigned, trajectories are resampled to 1 Hz, and the resulting seasonal dataset is saved for future runs.


In [ ]:
if preprocessed_path.exists() and not rebuild_preprocessed:
    print(f"Loading existing preprocessed dataset: {preprocessed_path}")
    trajs = Traffic(pd.read_parquet(preprocessed_path))

else:
    if rebuild_preprocessed and preprocessed_path.exists():
        print(f"Rebuilding preprocessed dataset: {preprocessed_path}")
    else:
        print("No preprocessed seasonal dataset found. Building it from raw data.")

    files = sorted(raw_dir.glob("*.parquet"))

    if not files:
        raise FileNotFoundError(f"No Parquet files found in {raw_dir}")

    print(f"Found {len(files)} raw-data files")

    df = pd.concat(
        (pd.read_parquet(file) for file in files),
        ignore_index=True,
    )

    trajs = Traffic(df)

    def convert_timestamp(value):
        if isinstance(value, (float, int)) and value > 1e9:
            return pd.to_datetime(value, unit="s", errors="coerce")
        if isinstance(value, pd.Timestamp):
            return value
        return pd.NaT

    trajs.data["timestamp"] = trajs.data["timestamp"].apply(convert_timestamp)
    trajs.data["onground"] = trajs.data["onground"].astype(np.bool_)

    trajs.data["latitude"] = trajs.data["latitude"].astype("double")
    trajs.data["longitude"] = trajs.data["longitude"].astype("double")
    trajs.data["groundspeed"] = trajs.data["groundspeed"].astype("double")
    trajs.data["track"] = trajs.data["track"].astype("double")
    trajs.data["vertical_rate"] = trajs.data["vertical_rate"].astype("double")
    trajs.data["altitude"] = trajs.data["altitude"].astype("double")
    trajs.data["geoaltitude"] = trajs.data["geoaltitude"].astype("double")

    trajs.data["icao24"] = trajs.data["icao24"].astype("object")
    trajs.data["callsign"] = trajs.data["callsign"].astype("object")
    trajs.data["squawk"] = trajs.data["squawk"].astype("object")

    trajs = trajs.assign_id().eval(
        max_workers=1,
        desc="Assigning flight IDs",
    )

    trajs = trajs.resample("1s").eval(
        max_workers=max_workers,
        desc="Resampling",
    )

    trajs.to_parquet(preprocessed_path)
    print(f"Saved preprocessed dataset: {preprocessed_path}")

trajs


## 4. Determine Departure Flights

Keep flights for which a departure runway can be identified at the selected airport.

In [ ]:
def dep_runway(flight):
    takeoff = flight.takeoff(
        airport_str,
        method="track_based",
        adv_filtering=True,
    )

    if takeoff is not None:
        next_segment = takeoff.next()
        if next_segment is not None:
            runway = next_segment.max("runway")
            return {
                "Flight ID": flight.flight_id,
                "Runway": runway,
            }


departure_runway = (
    trajs.iterate_lazy()
    .pipe(dep_runway)
    .eval(max_workers=max_workers, desc="Determining departure flights")
)

deps = trajs[departure_runway["Flight ID"].to_list()]

## 5. Remove Flights with Missing or Frozen Initial Track

Exclude flights without usable movement data or with a constant track during the first 0.06 NM of cumulative movement.

In [ ]:
def keep_valid_initial_track(flight):
    flight_with_distance = flight.cumulative_distance()
    initial_segment = flight_with_distance.query("cumdist <= 0.06")

    if (
        initial_segment is None
        or initial_segment.data is None
        or initial_segment.data.empty
    ):
        return None

    if initial_segment.data["track"].nunique() <= 1:
        return None

    return flight


deps_removed = (
    deps.iterate_lazy()
    .pipe(keep_valid_initial_track)
    .eval(max_workers=max_workers, desc="Filtering initial track")
)

## 6. Pushback Labeling

Identify the pushback segment from computed ground speed and track behavior.

In [ ]:
def pushback_complete(flight):
    # -------------------------------------------------------------------------
    # 1. Basic validation and preprocessing
    # -------------------------------------------------------------------------

    # Skip empty or invalid flight objects.
    if flight is None or flight.data is None or flight.data.empty:
        return None

    # Compute cumulative distance as well as position-derived ground speed
    # and ground track.
    flight = flight.cumulative_distance()

    # Absolute circular difference between the reported directional value
    # and the position-derived ground track.
    flight.data["track_difference_abs"] = abs(
        ((flight.data["track"] - flight.data["compute_track"] + 180) % 360) - 180
    )

    # Absolute change in position-derived ground track between consecutive points.
    flight.data["compute_track_diff_abs"] = abs(
        flight.data["compute_track"].diff()
    )

    # Absolute change in position-derived ground speed between consecutive points.
    flight.data["compute_gs_diff"] = abs(
        flight.data["compute_gs"].diff()
    )

    # Keep the complete trajectory for later validation and segment extraction.
    full_flight = flight

    # -------------------------------------------------------------------------
    # 2. Pushback candidate detection
    # -------------------------------------------------------------------------

    # Retain observations with a large difference between the reported
    # directional value and the position-derived ground track.
    flight = flight.query("track_difference_abs > 100")
    if flight is None or flight.data is None or flight.data.empty:
        return None

    # Restrict candidate observations to low-speed ground movement.
    flight = flight.query("compute_gs > 2.5")
    if flight is None or flight.data is None or flight.data.empty:
        return None

    flight = flight.query("compute_gs < 15")
    if flight is None or flight.data is None or flight.data.empty:
        return None

    # Exclude observations with large changes in position-derived ground track,
    # which are more likely caused by turns or noisy position reports.
    flight = flight.query("compute_track_diff_abs < 5")
    if flight is None or flight.data is None or flight.data.empty:
        return None

    # Use the median timestamp of all retained observations as the temporal
    # reference for the candidate pushback.
    median_timestamp = flight.data["timestamp"].median()

    # -------------------------------------------------------------------------
    # 3. Candidate validation
    # -------------------------------------------------------------------------

    # Extract a 60-second interval centred on the candidate reference time.
    candidate = full_flight.after(
        median_timestamp - pd.Timedelta("30s")
    ).first("60s")

    if candidate is None or candidate.data is None or candidate.data.empty:
        return None

    # Validate the candidate using all observations within the interval.
    # A sufficiently large median directional difference must still be present.
    median_track_diff_abs = candidate.data["track_difference_abs"].median()

    if median_track_diff_abs < 60:
        return None

    # -------------------------------------------------------------------------
    # 4. Pushback end detection
    # -------------------------------------------------------------------------

    # Identify observations where the position-derived ground speed is
    # effectively unchanged. These observations are used as a proxy for
    # stationary periods.
    stops = full_flight.query("compute_gs_diff < 9e-8")

    if stops is None or stops.data is None or stops.data.empty:
        return None

    # Only stationary periods after the pushback candidate reference time
    # are relevant for determining the pushback end.
    stops_after = stops.after(median_timestamp)

    if (
        stops_after is None
        or stops_after.data is None
        or stops_after.data.empty
    ):
        return None

    # Sort stationary observations chronologically and determine the temporal
    # gap between consecutive observations.
    df = stops_after.data.copy().sort_values("timestamp")
    df["dt"] = df["timestamp"].diff().dt.total_seconds().fillna(0)

    # Start a new stationary segment whenever two observations are separated
    # by more than 2 seconds.
    df["segment_id"] = (df["dt"] > 2).cumsum()

    # Determine start, end, and duration of each stationary segment.
    segments = (
        df.groupby("segment_id")["timestamp"]
        .agg(seg_start="min", seg_end="max")
        .reset_index()
    )

    segments["duration"] = (
        segments["seg_end"] - segments["seg_start"]
    ).dt.total_seconds()

    # Only stationary periods lasting at least 10 seconds are considered
    # potential pushback ends.
    valid = segments[segments["duration"] >= 10].reset_index(drop=True)

    if valid.empty:
        return None

    track_diff_threshold = 100

    def has_data(traffic):
        """Return True if a Traffic/Flight object contains observations."""
        return (
            traffic is not None
            and getattr(traffic, "data", None) is not None
            and not traffic.data.empty
        )

    # Default to the first valid stationary segment.
    chosen_idx = 0

    # If multiple stationary periods exist, determine whether an earlier
    # stationary period is only an intermediate stop during the pushback.
    for i in range(len(valid) - 1):
        ts_seg_end = valid.loc[i, "seg_end"]
        ts_next_start = valid.loc[i + 1, "seg_start"]

        # Guard against invalid or overlapping segment timestamps.
        if (
            ts_seg_end is None
            or ts_next_start is None
            or ts_next_start <= ts_seg_end
        ):
            chosen_idx = i
            break

        # Inspect the trajectory between two consecutive stationary periods.
        flight_between = full_flight.between(
            ts_seg_end,
            ts_next_start,
        )

        # If no observations exist between the two stationary periods,
        # accept the earlier segment as the pushback end.
        if not has_data(flight_between):
            chosen_idx = i
            break

        median_track_diff = flight_between.data[
            "track_difference_abs"
        ].median()

        # A large directional difference indicates continued pushback movement.
        # In this case, the earlier stationary period is treated as an
        # intermediate stop and the next candidate is evaluated.
        if (
            pd.notna(median_track_diff)
            and median_track_diff > track_diff_threshold
        ):
            continue

        # Otherwise, accept this stationary period as the pushback end.
        chosen_idx = i
        break

    else:
        # If all earlier stationary periods were rejected as intermediate
        # stops, use the final valid stationary period.
        chosen_idx = len(valid) - 1

    # The beginning of the selected stationary period defines the pushback end.
    ts_end = valid.loc[chosen_idx, "seg_start"]

    # Preliminary pushback segment: everything before the detected pushback end.
    pb_segment_candidate = full_flight.before(ts_end)

    duration = pb_segment_candidate.duration.total_seconds()

    # -------------------------------------------------------------------------
    # 5. Pushback start refinement
    # -------------------------------------------------------------------------

    # Short segments are retained directly because the available trajectory
    # may already begin after the actual pushback start.
    if duration < 30:
        return pb_segment_candidate

    # Estimate the initial aircraft position from the first 15 observations.
    start_segment = pb_segment_candidate.first("15")
    start_lat = start_segment.data["latitude"].median()
    start_lon = start_segment.data["longitude"].median()

    # Keep a copy of the preliminary pushback segment for the backward search.
    pb_segment_candidate_copy = pb_segment_candidate

    def haversine_m(lat1, lon1, lat2, lon2):
        """Calculate great-circle distance in metres."""
        earth_radius_m = 6_371_000

        lat1, lon1, lat2, lon2 = map(
            np.radians,
            [lat1, lon1, lat2, lon2],
        )

        dlat = lat2 - lat1
        dlon = lon2 - lon1

        a = (
            np.sin(dlat / 2) ** 2
            + np.cos(lat1)
            * np.cos(lat2)
            * np.sin(dlon / 2) ** 2
        )

        return 2 * earth_radius_m * np.arctan2(
            np.sqrt(a),
            np.sqrt(1 - a),
        )

    # Determine the distance of each observation from the estimated
    # initial aircraft position.
    radius_m = 30

    df = pb_segment_candidate.data.copy()

    distance = haversine_m(
        start_lat,
        start_lon,
        df["latitude"].values,
        df["longitude"].values,
    )

    # Retain observations more than 30 m from the initial position.
    pb_segment_candidate.data = df.loc[distance > radius_m]

    # Require a minimum position-derived ground speed to identify observations
    # associated with actual movement.
    pb_segment_candidate = pb_segment_candidate.query(
        "compute_gs > 0.5"
    )

    # If no suitable observations remain, keep the preliminary segment.
    if (
        pb_segment_candidate is None
        or pb_segment_candidate.data is None
        or pb_segment_candidate.data.empty
    ):
        return full_flight.before(ts_end)

    # Use the median timestamp of the retained movement observations as the
    # reference point for the backward search.
    median_start_time = pb_segment_candidate.data["timestamp"].median()

    # Remove observations for which the position-derived ground speed is
    # effectively unchanged.
    pb_segment_candidate_copy = pb_segment_candidate_copy.query(
        "compute_gs_diff > 8e-9"
    )

    # If no observations remain, keep the preliminary segment.
    if (
        pb_segment_candidate_copy is None
        or pb_segment_candidate_copy.data is None
        or pb_segment_candidate_copy.data.empty
    ):
        return full_flight.before(ts_end)

    # Prepare the remaining observations for the backward temporal-gap search.
    df = pb_segment_candidate_copy.data

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        utc=True,
        errors="coerce",
    )

    df = (
        df.dropna(subset=["timestamp"])
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    gap_s = 10
    median_ts = median_start_time

    # Only observations up to the temporal reference are relevant for
    # determining the start of the pushback.
    before = df.loc[df["timestamp"] <= median_ts].copy()

    if before.empty:
        pb_segment_candidate_copy.data = df

    else:
        # Traverse observations backwards from the temporal reference.
        backwards = (
            before.sort_values(
                "timestamp",
                ascending=False,
            )
            .reset_index(drop=True)
        )

        # Calculate the time difference to the next older observation.
        dt_back = (
            backwards["timestamp"]
            - backwards["timestamp"].shift(-1)
        ).dt.total_seconds()

        # Find the first gap larger than 10 seconds while moving backwards.
        gap_idx = np.where(dt_back.values > gap_s)[0]

        if gap_idx.size == 0:
            # If no gap exists, use the earliest remaining observation.
            cutoff_ts = backwards["timestamp"].min()

        else:
            # Use the chronologically newer observation immediately after
            # the first detected gap.
            cutoff_ts = backwards.loc[
                gap_idx[0],
                "timestamp",
            ]

        # Retain observations from the detected start region onwards.
        mask = (
            (df["timestamp"] > median_ts)
            | (df["timestamp"] >= cutoff_ts)
        )

        pb_segment_candidate_copy.data = (
            df.loc[mask].reset_index(drop=True)
        )

        # Shift the detected start 20 seconds backwards to recover observations
        # removed by the preceding movement filters.
        start_time = pb_segment_candidate_copy.start

        pb_segment_candidate = full_flight.between(
            start_time - pd.Timedelta("20s"),
            ts_end,
        )

    # Return the final automatically generated pushback segment.
    return pb_segment_candidate

## 7. Generate Pushback Candidates

In [ ]:
pb = (
    deps_removed.iterate_lazy()
    .pipe(pushback_complete)
    .eval(max_workers=max_workers, desc="Determining pushbacks")
)

no_pb = deps_removed - pb

## 8. Evaluate Pushback Candidates

Calculate the diagnostic metrics used for dataset-specific quality control.

In [ ]:
def evaluate_pushback(flight):
    duration = flight.duration.total_seconds()
    distance = flight.distance()
    distance_start = flight.data["cumdist"].iloc[0]
    average_speed = distance / duration

    return {
        "Flight ID": flight.flight_id,
        "Distance": distance,
        "Distance Start": distance_start,
        "Duration": duration,
        "Average Speed": average_speed,
        "Max Speed": flight.data["compute_gs"].max(),
        "Mean Speed": flight.data["compute_gs"].mean(),
        "Median Speed": flight.data["compute_gs"].median(),
        "Mean Speed Diff": flight.data["compute_gs_diff"].mean(),
        "Max Speed Diff": flight.data["compute_gs_diff"].max(),
    }


pb_evaluation = (
    pb.iterate_lazy()
    .pipe(evaluate_pushback)
    .eval(max_workers=max_workers, desc="Evaluating pushbacks")
)

pb_evaluation

## 9. Inspect Individual Pushback Labels

In [ ]:
def plot_flight(flight_id):
    df = deps_removed[flight_id].data

    pb_start = pb[flight_id].start
    pb_stop = pb[flight_id].stop
    pb_mask = (df["timestamp"] >= pb_start) & (df["timestamp"] <= pb_stop)

    df_normal = df[~pb_mask]
    df_pb = df[pb_mask]

    center_lat = df["latitude"].mean()
    center_lon = df["longitude"].mean()

    fig = go.Figure()

    fig.add_trace(
        go.Scattermap(
            lat=df_normal["latitude"],
            lon=df_normal["longitude"],
            mode="markers",
            marker=dict(size=6, color="steelblue"),
            text=df_normal["timestamp"].astype(str),
            hovertemplate=(
                "<b>timestamp:</b> %{text}<br>"
                "<b>lat:</b> %{lat:.5f}<br>"
                "<b>lon:</b> %{lon:.5f}<extra></extra>"
            ),
            name="Flight track",
        )
    )

    fig.add_trace(
        go.Scattermap(
            lat=df_pb["latitude"],
            lon=df_pb["longitude"],
            mode="markers",
            marker=dict(size=8, color="red"),
            text=df_pb["timestamp"].astype(str),
            hovertemplate=(
                "<b>PUSHBACK</b><br>"
                "<b>timestamp:</b> %{text}<br>"
                "<b>lat:</b> %{lat:.5f}<br>"
                "<b>lon:</b> %{lon:.5f}<extra></extra>"
            ),
            name="Pushback segment",
        )
    )

    fig.update_layout(
        map=dict(
            style="carto-positron",
            center=dict(lat=center_lat, lon=center_lon),
            zoom=12,
        ),
        margin=dict(l=0, r=0, t=30, b=0),
        title=f"Flight {flight_id}",
        legend=dict(x=0.01, y=0.99),
    )

    fig.show()

## 10. Apply Airport- and Season-Specific Pushback Filters

Automatic thresholds are read from the YAML configuration. Set a threshold to `null` in a configuration file to disable that filter for a specific airport/season combination.

In [ ]:
filters = review_config.get("filters", {})
manual_review = review_config.get("manual_review", {})

pb_to_remove = []

max_distance = filters.get("max_distance")
if max_distance is not None:
    pb_to_remove += pb_evaluation.loc[
        pb_evaluation["Distance"] > max_distance,
        "Flight ID",
    ].to_list()

max_distance_start = filters.get("max_distance_start")
if max_distance_start is not None:
    pb_to_remove += pb_evaluation.loc[
        pb_evaluation["Distance Start"] > max_distance_start,
        "Flight ID",
    ].to_list()

min_duration = filters.get("min_duration")
if min_duration is not None:
    pb_to_remove += pb_evaluation.loc[
        pb_evaluation["Duration"] < min_duration,
        "Flight ID",
    ].to_list()

min_average_speed = filters.get("min_average_speed")
if min_average_speed is not None:
    pb_to_remove += pb_evaluation.loc[
        pb_evaluation["Average Speed"] < min_average_speed,
        "Flight ID",
    ].to_list()

pb_to_remove += manual_review.get("pushbacks_remove", [])

pushbacks_keep = set(manual_review.get("pushbacks_keep", []))
pb_to_remove = sorted(set(pb_to_remove) - pushbacks_keep)

print(f"Pushback flights marked for removal: {len(pb_to_remove)}")
pb_to_remove

## 11. Review Flights with No Detected Pushback

Use the widget to inspect flights for which no pushback was detected. False negatives identified during this review are stored in the YAML configuration under `manual_review.no_pushbacks_remove`.

In [ ]:
flights = list(no_pb)

batch_size = 10
current_idx = 0

btn_next = widgets.Button(description="▶ Next 10", button_style="primary")
btn_prev = widgets.Button(
    description="◀ Previous 10",
    button_style="warning",
    disabled=True,
)
btn_label = widgets.Label()

button_box = widgets.HBox([btn_prev, btn_next, btn_label])
output = widgets.Output()


def update_label():
    total = len(flights)
    start_idx = current_idx - batch_size
    end_idx = min(start_idx + batch_size, total)
    btn_label.value = f"Flights {start_idx + 1}–{end_idx} of {total}"


def update_buttons():
    btn_prev.disabled = current_idx <= batch_size
    btn_next.disabled = current_idx >= len(flights)


def plot_batch(start_idx):
    fig = go.Figure()
    batch = flights[start_idx : start_idx + batch_size]

    latitudes = []
    longitudes = []

    for flight in batch:
        latitudes.extend(flight.data["latitude"].dropna().to_list())
        longitudes.extend(flight.data["longitude"].dropna().to_list())

        fig.add_trace(
            go.Scattermap(
                mode="lines",
                lat=flight.data.latitude,
                lon=flight.data.longitude,
                line=dict(width=2),
                opacity=0.7,
                name=flight.flight_id,
            )
        )

    center_lat = np.mean(latitudes) if latitudes else 0
    center_lon = np.mean(longitudes) if longitudes else 0

    fig.update_layout(
        width=1000,
        height=600,
        margin=dict(l=0, r=0, t=0, b=0),
        map=dict(
            style="carto-positron",
            zoom=13,
            center=dict(lat=center_lat, lon=center_lon),
        ),
    )

    with output:
        clear_output(wait=True)
        fig.show()


def on_next(_):
    global current_idx
    plot_batch(current_idx)
    current_idx = min(current_idx + batch_size, len(flights))
    update_label()
    update_buttons()


def on_prev(_):
    global current_idx
    current_idx = max(current_idx - 2 * batch_size, 0)
    plot_batch(current_idx)
    current_idx = min(current_idx + batch_size, len(flights))
    update_label()
    update_buttons()


btn_next.on_click(on_next)
btn_prev.on_click(on_prev)

display(button_box, output)

plot_batch(0)
current_idx = batch_size
update_label()
update_buttons()

## 12. Apply Manual Review Decisions

Combine all pushback exclusions with manually identified false negatives from the selected airport/season configuration.

In [ ]:
no_pbs_to_remove = manual_review.get("no_pushbacks_remove", [])

ids_to_remove = sorted(set(pb_to_remove + no_pbs_to_remove))

if ids_to_remove:
    removed_flights = deps_removed[ids_to_remove]
    deps_removed_final = deps_removed - removed_flights
else:
    removed_flights = None
    deps_removed_final = deps_removed

print(f"Total flights removed after review: {len(ids_to_remove)}")

## 13. Store Reviewed Labels

Create the final mapping between raw flight IDs and pushback labels after all automatic and manual quality-control steps.

In [ ]:
def create_mapping(flight):
    pushback = pushback_complete(flight)

    has_pushback = False
    pushback_start = pd.NaT
    pushback_stop = pd.NaT

    if pushback:
        has_pushback = True
        pushback_start = pushback.start
        pushback_stop = pushback.stop

    return {
        "Flight ID Raw": flight.flight_id,
        "Callsign": flight.callsign,
        "Start": flight.start,
        "Stop": flight.stop,
        "Pushback": has_pushback,
        "Pushback Start": pushback_start,
        "Pushback Stop": pushback_stop,
    }


mapping_set = (
    deps_removed_final.iterate_lazy()
    .pipe(create_mapping)
    .eval(max_workers=max_workers, desc="Creating label mapping")
)

mapping_path = reviewed_dir / f"{airport_str}_{season}_mapping_set.csv"
mapping_set.to_csv(mapping_path, index=False)

mapping_set

## 14. Store Excluded Flights

Store all excluded flights separately to keep the review process traceable.

In [ ]:
def create_removed_mapping(flight):
    return {
        "Flight ID Raw": flight.flight_id,
        "Callsign": flight.callsign,
        "Start": flight.start,
        "Stop": flight.stop,
    }


removed_path = (
    reviewed_dir
    / f"{airport_str}_{season}_excluded_flights.csv"
)

if removed_flights is not None:
    mapping_set_removed = (
        removed_flights.iterate_lazy()
        .pipe(create_removed_mapping)
        .eval(
            max_workers=max_workers,
            desc="Mapping excluded flights",
        )
    )

    mapping_set_removed.to_csv(
        removed_path,
        index=False,
    )

    mapping_set_removed

else:
    print("No flights were excluded.")